## This program inferring UML/OCL representations by LLM4Models LLM   

In [ ]:
"""
Author: Hanan Abdulwahab Siala
University: King's College London
Date: 2020-10-05

Description:
    This program infers UML class diagrams and OCL specifications from both Java and Python programs using the LLM4Models LLM.
"""

In [ ]:
# ----------------------------------------------------------------------------
# Please execute preprocessing program first to remove comments 

# Put your program in Test1.java for Java programs or Test1.py for Python programs.
# The output will be saved in LLM4Models.txt

# For What_I_Want variable:
#    - Choose 1 to abstract UML from Java
#    - Choose 2 to abstract UML from Python
#    - Choose 3 to abstract OCL from Java
#    - Choose 4 to abstract OCL from Python

# Full_Model=True     for using a full model
# Full_Model=False    for using LoRA adapter

# Choose version 1, 2, 3, 4 for UML ( 4 for Java-UML only ).
# Choose version 1, 2 for OCL.
# ----------------------------------------------------------------------------
What_I_Want=1
Full_Model=False
Version=2
# ----------------------------------------------------------------------------

In [ ]:
import time
import torch
import os

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login

In [ ]:
DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
token = "YOUR_TOKEN_HERE" 
login(token=token)

In [ ]:
# ----------------------------------------------------------------------------
# choose 1 for version 1, 2 for version 2, 3 for version 3.
# choose 4 for version 4 for Java-UML only.

if Full_Model==False:
   if What_I_Want==1:
      if Version==1: 
         CheckPoint='HA-Siala/Java-UML-v0.1' 
      elif Version==2:    
         CheckPoint='HA-Siala/Java-UML-v0.2' 
      elif Version==3:    
         CheckPoint='HA-Siala/Java-UML-v0.3' 
      else:    
         CheckPoint='HA-Siala/Java-UML-v0.4' 
   elif What_I_Want==2:
      if Version==1:   
         CheckPoint='HA-Siala/Python-UML-v0.1' 
      elif Version==2:    
         CheckPoint='HA-Siala/Python-UML-v0.2'    
      else:    
         CheckPoint='HA-Siala/Python-UML-v0.3' 
   elif What_I_Want==3:
      if Version==1:        
         CheckPoint='HA-Siala/Java-OCL-v0.1' 
      else:    
         CheckPoint='HA-Siala/Java-OCL-v0.2' 
   else:
      if Version==1:        
         CheckPoint='HA-Siala/Python-OCL-v0.1' 
      else:    
         CheckPoint='HA-Siala/Python-OCL-v0.2'
    
   mistral_checkpoint = "mistralai/Mistral-7B-v0.3"
   tokenizer = AutoTokenizer.from_pretrained(mistral_checkpoint, use_fast=True) 
   tokenizer.pad_token = tokenizer.unk_token   
   tokenizer.padding_side = "left" 
   model = AutoModelForCausalLM.from_pretrained(mistral_checkpoint, torch_dtype=torch.bfloat16, device_map="auto") 
   model = PeftModel.from_pretrained(model, CheckPoint, torch_dtype=torch.bfloat16, is_trainable=False) 
else:
   if What_I_Want==1:
      if Version==1:       
         CheckPoint='HA-Siala/Java-UML-full-v0.1' 
      elif Version==2:       
         CheckPoint='HA-Siala/Java-UML-full-v0.2' 
      else:    
         CheckPoint='HA-Siala/Java-UML-full-v0.3' 
   elif What_I_Want==2:
      if Version==1: 
         CheckPoint='HA-Siala/Python-UML-full-v0.1' 
      elif Version==2: 
         CheckPoint='HA-Siala/Python-UML-full-v0.2' 
      else:    
         CheckPoint='HA-Siala/Python-UML-full-v0.3' 
   elif What_I_Want==3:
      if Version==1:        
         CheckPoint='HA-Siala/Java-OCL-full-v0.1' 
      else:    
         CheckPoint='HA-Siala/Java-OCL-full-v0.2' 
   else:
      if Version==1:        
         CheckPoint='HA-Siala/Python-OCL-full-v0.1' 
      else:    
         CheckPoint='HA-Siala/Python-OCL-full-v0.2' 
    
   model = AutoModelForCausalLM.from_pretrained(CheckPoint, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True) 
   tokenizer = AutoTokenizer.from_pretrained(CheckPoint, use_fast=True)
   tokenizer.pad_token = tokenizer.unk_token   
   tokenizer.padding_side = "left" 
model.eval() 
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def format_time(seconds):
   h = int(seconds // 3600)
   m = int((seconds % 3600) // 60)
   s = seconds % 60
   result = ""
   if h > 0:
      result += f"{h}h "
   if m > 0 or h > 0:  
      result += f"{m}m "
   result += f"{s:.6f}s"
   return result
# ----------------------------------------------------------------------------   
def GeneratePrompt(content):
   if What_I_Want==1:
      Instruction="""Generate a concise UML class diagram for the provided Java code. The output should:
1. Define each class and interface only once, including its attributes, methods, and relationships.
2. Include all relationships (Inheritance, Realization, Dependency, Association, Composition, Aggregation) without duplication.
3. Avoid redundant or repeated operations, classes, or relationships."""
   elif What_I_Want==2:
      Instruction="""Generate a concise UML class diagram for the provided Python code. The output should:
1. Define each class and interface only once, including its attributes, methods, and relationships.
2. Include all relationships (Inheritance, Realization, Dependency, Association, Composition, Aggregation) without duplication.
3. Avoid redundant or repeated operations, classes, or relationships."""
   elif What_I_Want==3:
      Instruction="""Generate an Object Constraint Language (OCL) specification for the provided Java code. The output should:
1. Ensure no repeated or redundant operations or classes.
2. Include only the OCL code for the provided Java code.
3. Do not include statements for items not found in the Java code."""
   else:
      Instruction="""Generate an Object Constraint Language (OCL) specification for the provided Python code. The output should:
1. Ensure no repeated or redundant operations or classes.
2. Include only the OCL code for the provided Python code.
3. Do not include statements for items not found in the Python code."""

   prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response, which is in JSON format that appropriately solves the following Task:

### Instruction:
{Instruction}

### Input:
{content}

### Response:
"""   
   return prompt
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def GenerateInferenceOutput(text): 
   tokenizer.pad_token = tokenizer.unk_token  
   inputs = tokenizer(GeneratePrompt(text), return_tensors="pt").to(DEVICE)
   InputTokens = inputs["input_ids"].shape[1] 
   torch.cuda.empty_cache() # new 
   with torch.inference_mode():
      outputs = model.generate(
         **inputs,  
         max_new_tokens= 32768, 
         temperature= 0.2, 
         do_sample= True, 
         pad_token_id=tokenizer.eos_token_id,
         top_p=0.9,
      )
      output_p = tokenizer.batch_decode(outputs, skip_special_tokens=True)
      if output_p:
         output_text = output_p[0]
         split_text_p = output_text.split("Response:")
         if len(split_text_p) > 1:
            cleaned = split_text_p[1] 
            cleaned = cleaned.split("###")[0].strip() 
            return cleaned, InputTokens
         else:
            return None, InputTokens
      else:
         return None, 0 
# ----------------------------------------------------------------------------       

In [ ]:
# ----------------------------------------------------------------------------
def ReadFile(FilePath):
   if not os.path.exists(FilePath):
      print(f"Error: File '{FilePath}' does not exist.")
      return None
   if (What_I_Want in [1,3] and not FilePath.endswith('.java')) or (What_I_Want in [2,4] and not FilePath.endswith('.py')):
      print(f"Warning: '{FilePath}' may not be a program file.")
   with open(FilePath, 'r', encoding='utf-8') as file:
      content = file.read()
      return content
if What_I_Want in [1,3]:
   FilePath = "Test1.java" 
else:
   FilePath = "Test1.py"
content = ReadFile(FilePath)
if content:
   print("Successfully read file!")
   print() 
   StartTime=time.time()
   Output, InputTokens = GenerateInferenceOutput(content)
   EndTime=time.time()
   InferenceTime=EndTime - StartTime
   TimePerInputToken = InferenceTime / InputTokens

   LLM4Models = "LLM4Models.txt"
   with open(LLM4Models, "w", encoding="utf-8") as target_file:
      target_file.write(Output)
      print(f"The output is saved to {LLM4Models}\n")
   #print(Output)
   #print()
   print(f"Inference Time: {format_time(InferenceTime)}")
   print(f"Execution time per input token: {format_time(TimePerInputToken)}")
# ----------------------------------------------------------------------------